##  Load Data

In [1]:
import pandas as pd

df = pd.read_csv('data/synthetic_fraud_data.csv')
print(f'Shape: {df.shape}')
print(f'Fraud rate: {df["is_fraud"].mean()*100:.2f}%')
df.head(3)

Shape: (7483766, 24)
Fraud rate: 19.97%


,transaction_id,customer_id,card_number,timestamp,merchant_category,merchant_type,merchant,amount,currency,country,...,device,channel,device_fingerprint,ip_address,distance_from_home,high_risk_merchant,transaction_hour,weekend_transaction,velocity_last_hour,is_fraud
0,TX_a0ad2a2a,CUST_72886,6646734767813109,2024-09-30 00:00:01.034820+00:00,Restaurant,fast_food,Taco Bell,294.87,GBP,UK,...,iOS App,mobile,e8e6160445c935fd0001501e4cbac8bc,197.153.60.199,0,False,0,False,"{'num_transactions': 1197, 'total_amount': 334...",False
1,TX_3599c101,CUST_70474,376800864692727,2024-09-30 00:00:01.764464+00:00,Entertainment,gaming,Steam,3368.97,BRL,Brazil,...,Edge,web,a73043a57091e775af37f252b3a32af9,208.123.221.203,1,True,0,False,"{'num_transactions': 509, 'total_amount': 2011...",True
2,TX_a9461c6d,CUST_10715,5251909460951913,2024-09-30 00:00:02.273762+00:00,Grocery,physical,Whole Foods,102582.38,JPY,Japan,...,Firefox,web,218864e94ceaa41577d216b149722261,10.194.159.204,0,False,0,False,"{'num_transactions': 332, 'total_amount': 3916...",False


In [2]:
df.columns

Index(['transaction_id', 'customer_id', 'card_number', 'timestamp',
       'merchant_category', 'merchant_type', 'merchant', 'amount', 'currency',
       'country', 'city', 'city_size', 'card_type', 'card_present', 'device',
       'channel', 'device_fingerprint', 'ip_address', 'distance_from_home',
       'high_risk_merchant', 'transaction_hour', 'weekend_transaction',
       'velocity_last_hour', 'is_fraud'],
      dtype='object')

##  Feature Engineering



In [2]:
df.drop("transaction_id", axis=1, inplace=True)

In [3]:
df["customer_tx_count"] = df.groupby("customer_id")["customer_id"].transform("count")

df["customer_avg_amount"] = df.groupby("customer_id")["amount"].transform("mean")


In [4]:
df.drop("customer_id", axis=1, inplace=True)

In [5]:
df["card_tx_count"] = df.groupby("card_number")["card_number"].transform("count")
df["card_avg_amount"] = df.groupby("card_number")["amount"].transform("mean")

In [6]:
df.drop("card_number", axis=1, inplace=True)

In [7]:
df["timestamp"] = pd.to_datetime(df["timestamp"], format="ISO8601", errors="coerce")
df["hour"] = df["timestamp"].dt.hour
df["weekday"] = df["timestamp"].dt.weekday
df["night_transaction"] = (df["hour"] < 6).astype(int)

In [8]:
df.drop("timestamp", axis=1, inplace=True)

In [9]:
df.drop("transaction_hour", axis=1, inplace=True)

In [10]:
df.drop("weekend_transaction", axis=1, inplace=True)

In [13]:
df.groupby("merchant_category")["is_fraud"].mean().sort_values(ascending=False)

merchant_category
Travel           0.200341
Grocery          0.200194
Gas              0.199731
Restaurant       0.199696
Entertainment    0.199632
Education        0.199459
Retail           0.199398
Healthcare       0.199376
Name: is_fraud, dtype: float64

In [11]:
df.drop("merchant_category", axis=1, inplace=True)

In [15]:
df.groupby("merchant_type")["is_fraud"].mean().sort_values(ascending=False)

merchant_type
transport    0.201169
fast_food    0.200719
airlines     0.200535
physical     0.200206
booking      0.200174
events       0.200017
major        0.199821
pharmacy     0.199773
supplies     0.199758
local        0.199642
gaming       0.199558
hotels       0.199487
streaming    0.199320
online       0.199310
casual       0.199186
premium      0.199179
medical      0.198978
Name: is_fraud, dtype: float64

In [12]:
df.drop("merchant_type", axis=1, inplace=True)

In [13]:
df["merchant_tx_count"] = df.groupby("merchant")["merchant"].transform("count")

In [14]:
df.drop("merchant", axis=1, inplace=True)

In [15]:
import numpy as np

df["amount_log"] = np.log1p(df["amount"])

In [16]:
df["currency"].value_counts()

currency
EUR    1065751
NGN     849840
BRL     804800
RUB     793730
MXN     785704
SGD     588668
GBP     538493
CAD     532632
JPY     527393
USD     500060
AUD     496695
Name: count, dtype: int64

In [16]:
df.groupby("country")["is_fraud"].mean().sort_values(ascending=False)

country
Mexico       0.380348
Russia       0.377238
Brazil       0.371060
Nigeria      0.351360
Australia    0.075805
USA          0.074615
Japan        0.071279
Germany      0.070939
Canada       0.069988
UK           0.069351
France       0.069143
Singapore    0.063557
Name: is_fraud, dtype: float64

In [23]:
df.groupby("city")["is_fraud"].mean().sort_values(ascending=False)

city
Unknown City    0.208687
Los Angeles     0.076191
San Jose        0.075517
Phoenix         0.075219
Philadelphia    0.074909
San Diego       0.074784
San Antonio     0.074602
New York        0.074209
Chicago         0.074151
Houston         0.073803
Dallas          0.072777
Name: is_fraud, dtype: float64

In [17]:
df.groupby("city_size")["is_fraud"].mean().sort_values(ascending=False)

city_size
medium    0.203150
large     0.074585
Name: is_fraud, dtype: float64

In [18]:
country_fraud_rate = df.groupby("country")["is_fraud"].mean()

df["country_risk"] = df["country"].map(country_fraud_rate)#risk score

In [19]:
df.drop("country", axis=1, inplace=True)

In [20]:
df["city_unknown"] = (df["city"] == "Unknown City").astype(int)#ken uknown rahi suspicious khater snn kifkif

In [21]:
df.drop("city", axis=1, inplace=True)

In [22]:
df["city_size"] = df["city_size"].map({
    "large": 0,
    "medium": 1
})

In [30]:
df.groupby("device")["is_fraud"].mean().sort_values(ascending=False)

device
Chip Reader        1.000000
Magnetic Stripe    1.000000
NFC Payment        1.000000
Firefox            0.126831
Safari             0.126204
Android App        0.125070
Chrome             0.123710
iOS App            0.122703
Edge               0.116753
Name: is_fraud, dtype: float64

In [23]:
device_stats = df.groupby("device")["is_fraud"].agg(["count", "mean"]).sort_values("mean", ascending=False)
print(device_stats)

                   count      mean
device                            
Chip Reader       217324  1.000000
Magnetic Stripe   217204  1.000000
NFC Payment       216519  1.000000
Firefox          1120952  0.126831
Safari           1120245  0.126204
Android App      1126117  0.125070
Chrome           1132384  0.123710
iOS App          1143461  0.122703
Edge             1189560  0.116753


In [47]:
high_risk_devices = ["Chip Reader", "Magnetic Stripe", "NFC Payment"]

df["device_high_risk"] = df["device"].isin(high_risk_devices).astype("int8")

In [48]:
df.drop(columns=["device"], inplace=True)

In [24]:
df["transactions_per_device"] = (
    df.groupby("device_fingerprint")["device_fingerprint"]
      .transform("count")
      .astype("int32")
)

In [25]:
df[["transactions_per_device", "is_fraud"]].corr()

,transactions_per_device,is_fraud
transactions_per_device,1.000000,-0.652837
is_fraud,-0.652837,1.000000


In [25]:
df["ip_transaction_count"] = (
    df.groupby("ip_address")["ip_address"]
      .transform("count")
      .astype("int32")
)

In [26]:
df.drop(columns=["ip_address"], inplace=True)

In [40]:
df.groupby("card_type")["is_fraud"].mean().sort_values(ascending=False)

card_type
Basic Credit       0.199742
Platinum Credit    0.199731
Gold Credit        0.199729
Premium Debit      0.199721
Basic Debit        0.199720
Name: is_fraud, dtype: float64

In [41]:
df.groupby("card_present")["is_fraud"].mean().sort_values(ascending=False)

card_present
True     1.000000
False    0.123475
Name: is_fraud, dtype: float64

In [27]:
df.drop(columns=["card_type"], inplace=True)

In [28]:
df.drop(columns=["card_present"], inplace=True)#ntesti beha

In [29]:
df["card_present"] = df["card_present"].astype("int8")#baad chntesti ble beha

KeyError: 'card_present'

In [44]:
df.groupby("high_risk_merchant")["is_fraud"].mean().sort_values(ascending=False)

high_risk_merchant
True     0.199986
False    0.199642
Name: is_fraud, dtype: float64

In [30]:
df.drop(columns=["high_risk_merchant"], inplace=True)

In [31]:
import ast

df["velocity_last_hour"] = df["velocity_last_hour"].apply(ast.literal_eval)

In [48]:
df["velocity_last_hour"].iloc[0]

{'num_transactions': 1197,
 'total_amount': 33498556.080464985,
 'unique_merchants': 105,
 'unique_countries': 12,
 'max_single_amount': 1925480.6324148502}

In [32]:
velocity_features = pd.json_normalize(df["velocity_last_hour"])

In [33]:
velocity_features.columns = [
    "velocity_tx_last_hour",
    "velocity_amount_last_hour",
    "velocity_unique_merchants",
    "velocity_unique_countries",
    "velocity_max_single_amount"
]

In [34]:
df = pd.concat([df, velocity_features], axis=1)

In [35]:
df.drop(columns=["velocity_last_hour"], inplace=True)

In [36]:
df[
    [
        "velocity_tx_last_hour",
        "velocity_amount_last_hour",
        "velocity_unique_merchants",
        "velocity_unique_countries",
        "velocity_max_single_amount",
        "is_fraud"
    ]
].corr()["is_fraud"].sort_values(ascending=False)

is_fraud                      1.000000
velocity_max_single_amount    0.009226
velocity_unique_countries     0.009046
velocity_unique_merchants     0.006932
velocity_tx_last_hour         0.004506
velocity_amount_last_hour     0.003332
Name: is_fraud, dtype: float64

In [37]:
df.drop(columns=[
    "velocity_tx_last_hour",
    "velocity_amount_last_hour",
    "velocity_unique_merchants",
    "velocity_unique_countries",
    "velocity_max_single_amount"
], inplace=True)

In [55]:
df.groupby("channel")["is_fraud"].mean().sort_values(ascending=False)

channel
pos       1.000000
mobile    0.123878
web       0.123275
Name: is_fraud, dtype: float64

In [56]:
df.groupby("channel")["is_fraud"].agg(["count","mean"]).sort_values("mean", ascending=False)

,count,mean
channel,,
pos,651047,1.000000
mobile,2269578,0.123878
web,4563141,0.123275


In [38]:
df["pos_transaction"] = (df["channel"] == "pos").astype("int8")

In [39]:
df.drop(columns=["channel"], inplace=True)

In [59]:
df.columns

Index(['amount', 'currency', 'city_size', 'card_present', 'device_fingerprint',
       'distance_from_home', 'is_fraud', 'customer_tx_count',
       'customer_avg_amount', 'card_tx_count', 'card_avg_amount', 'hour',
       'weekday', 'night_transaction', 'merchant_tx_count', 'amount_log',
       'country_risk', 'city_unknown', 'device_high_risk',
       'transactions_per_device', 'ip_transaction_count', 'pos_transaction'],
      dtype='object')

In [40]:
df.drop(columns=["device_fingerprint"], inplace=True)

In [41]:
df = pd.get_dummies(df, columns=["currency"], drop_first=True, dtype="int8")

##  Prepare Features 

In [49]:
X = df.drop("is_fraud", axis=1)
y = df["is_fraud"]

In [50]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

##  Isolation Forest 


In [73]:
from sklearn.ensemble import IsolationForest

model = IsolationForest(
    n_estimators=70,        # less trees = less memory
    max_samples=100000,     # explicit subsampling per tree
    contamination=0.2,
    random_state=42,
    n_jobs=1                # important: avoid parallel memory explosion
)



In [45]:
for col in X_train.columns:
    if X_train[col].dtype == "float64":
        X_train[col] = X_train[col].astype("float32")
        X_test[col] = X_test[col].astype("float32")
    elif X_train[col].dtype == "int64":
        X_train[col] = X_train[col].astype("int32")
        X_test[col] = X_test[col].astype("int32")
    elif X_train[col].dtype == "bool":
        X_train[col] = X_train[col].astype("int8")
        X_test[col] = X_test[col].astype("int8")

In [74]:
model.fit(X_train)

c:\Users\eyaha\anaconda3\envs\Atelier_ML\lib\site-packages\sklearn\base.py:450: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(


IsolationForest(contamination=0.2, max_samples=100000, n_estimators=70,
                n_jobs=1, random_state=42)

In [75]:
y_pred = model.predict(X_test)


In [76]:
y_pred = (y_pred == -1).astype(int)


In [78]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy : 0.822214605740155
Precision: 0.5547944062757555
Recall   : 0.5561844358809677
F1-score : 0.5554885514928077

Confusion Matrix:
[[1064385  133425]
 [ 132676  166268]]

Classification Report:
              precision    recall  f1-score   support

       False       0.89      0.89      0.89   1197810
        True       0.55      0.56      0.56    298944

    accuracy                           0.82   1496754
   macro avg       0.72      0.72      0.72   1496754
weighted avg       0.82      0.82      0.82   1496754



In [ ]:
from sklearn.ensemble import IsolationForest

model1 = IsolationForest(
    n_estimators=70,        
    max_samples=100000,     
    contamination=0.18,
    random_state=42,
    n_jobs=1                
)



In [81]:
model1.fit(X_train)

c:\Users\eyaha\anaconda3\envs\Atelier_ML\lib\site-packages\sklearn\base.py:450: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(


IsolationForest(contamination=0.18, max_samples=100000, n_estimators=70,
                n_jobs=1, random_state=42)

In [ ]:
y_pred = model1.predict(X_test)


In [85]:
y_pred = (y_pred == -1).astype(int)


In [86]:
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy : 0.824748756308652
Precision: 0.5679178963916532
Recall   : 0.5123769000214087
F1-score : 0.5387196387245626

Confusion Matrix:
[[1081274  116536]
 [ 145772  153172]]

Classification Report:
              precision    recall  f1-score   support

       False       0.88      0.90      0.89   1197810
        True       0.57      0.51      0.54    298944

    accuracy                           0.82   1496754
   macro avg       0.72      0.71      0.72   1496754
weighted avg       0.82      0.82      0.82   1496754



In [87]:
model1.fit(X_train[y_train == 0])

c:\Users\eyaha\anaconda3\envs\Atelier_ML\lib\site-packages\sklearn\base.py:450: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(


IsolationForest(contamination=0.18, max_samples=100000, n_estimators=70,
                n_jobs=1, random_state=42)

In [88]:
y_pred = model1.predict(X_test)


In [89]:
y_pred = (y_pred == -1).astype(int)


In [90]:
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy : 0.8405963839081105
Precision: 0.5613780675813649
Recall   : 0.9233000160565189
F1-score : 0.6982259447672579

Confusion Matrix:
[[982151 215659]
 [ 22929 276015]]

Classification Report:
              precision    recall  f1-score   support

       False       0.98      0.82      0.89   1197810
        True       0.56      0.92      0.70    298944

    accuracy                           0.84   1496754
   macro avg       0.77      0.87      0.79   1496754
weighted avg       0.89      0.84      0.85   1496754



In [93]:
from sklearn.ensemble import IsolationForest

model2 = IsolationForest(
    n_estimators=100,        # less trees = less memory
    max_samples=100000,     # explicit subsampling per tree
    contamination=0.15,
    random_state=42,
    n_jobs=1                # important: avoid parallel memory explosion
)



In [94]:
model2.fit(X_train)

c:\Users\eyaha\anaconda3\envs\Atelier_ML\lib\site-packages\sklearn\base.py:450: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(


IsolationForest(contamination=0.15, max_samples=100000, n_jobs=1,
                random_state=42)

In [ ]:
y_pred = model2.predict(X_test)


In [96]:
y_pred = (y_pred == -1).astype(int)


In [97]:
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy : 0.830016154959332
Precision: 0.5991757629761639
Recall   : 0.449866864697067
F1-score : 0.5138958413738025

Confusion Matrix:
[[1107845   89965]
 [ 164459  134485]]

Classification Report:
              precision    recall  f1-score   support

       False       0.87      0.92      0.90   1197810
        True       0.60      0.45      0.51    298944

    accuracy                           0.83   1496754
   macro avg       0.73      0.69      0.71   1496754
weighted avg       0.82      0.83      0.82   1496754



In [98]:
model2.fit(X_train[y_train == 0])

c:\Users\eyaha\anaconda3\envs\Atelier_ML\lib\site-packages\sklearn\base.py:450: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(


IsolationForest(contamination=0.15, max_samples=100000, n_jobs=1,
                random_state=42)

In [99]:
y_pred = model2.predict(X_test)


In [100]:
y_pred = (y_pred == -1).astype(int)


In [101]:
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy : 0.8585225093769584
Precision: 0.597564551035435
Recall   : 0.8931505566259902
F1-score : 0.7160524618476963

Confusion Matrix:
[[1017995  179815]
 [  31942  267002]]

Classification Report:
              precision    recall  f1-score   support

       False       0.97      0.85      0.91   1197810
        True       0.60      0.89      0.72    298944

    accuracy                           0.86   1496754
   macro avg       0.78      0.87      0.81   1496754
weighted avg       0.90      0.86      0.87   1496754



In [51]:
from sklearn.ensemble import IsolationForest

model2 = IsolationForest(
    n_estimators=100,        # less trees = less memory
    max_samples=100000,     # explicit subsampling per tree
    contamination=0.15,
    random_state=42,
    n_jobs=1                # important: avoid parallel memory explosion
)


model2.fit(X_train[y_train == 0])

KeyboardInterrupt: 

In [45]:
y_pred = model2.predict(X_test)

In [46]:
y_pred = (y_pred == -1).astype(int)


In [48]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy : 0.8534181301670147
Precision: 0.5907178096439348
Recall   : 0.8663462053093556
F1-score : 0.7024621122224105

Confusion Matrix:
[[1018368  179442]
 [  39955  258989]]

Classification Report:
              precision    recall  f1-score   support

       False       0.96      0.85      0.90   1197810
        True       0.59      0.87      0.70    298944

    accuracy                           0.85   1496754
   macro avg       0.78      0.86      0.80   1496754
weighted avg       0.89      0.85      0.86   1496754



In [52]:
from sklearn.ensemble import IsolationForest

model3 = IsolationForest(
    n_estimators=100,        # less trees = less memory
    max_samples=100000,     # explicit subsampling per tree
    contamination=0.15,
    random_state=42,
    n_jobs=1                # important: avoid parallel memory explosion
)


model3.fit(X_train[y_train == 0])

c:\Users\eyaha\anaconda3\envs\Atelier_ML\lib\site-packages\sklearn\base.py:450: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(


IsolationForest(contamination=0.15, max_samples=100000, n_jobs=1,
                random_state=42)

In [53]:
y_pred = model3.predict(X_test)

In [54]:
y_pred = (y_pred == -1).astype(int)


In [55]:
import joblib

joblib.dump(model3, "model.joblib")


['model.joblib']

In [56]:
feature_names = list(X_train.columns)


In [57]:
import json

with open("metadata.json", "w") as f:
    json.dump({
        "model_name": "fraud-isolation-forest",
        "model_version": "v1",
        "threshold": 0.5,
        "feature_order": feature_names
    }, f)